# 📰  Pydantic 파이프라인 + 뉴스 모니터링

```
Phase 0~4. Core 학습 — Pydantic + RunnableParallel 핵심 부품
Project.   기사 → 분석 + 요약 + 태깅 + 대응 라우팅
```

## Phase 0. 환경 설정

In [1]:
%pip install -q langchain langchain-openai python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

✅ 로컬 Jupyter
✅ API 키 OK


## Phase 1. PydanticOutputParser — 자연어를 객체로

5단계 패턴: 스키마 → 파서 → 프롬프트 → LLM → 체인

In [3]:
class Summary(BaseModel):
    title: str = Field(description="짧은 제목")
    sentiment: Literal["positive", "negative", "neutral"]
    key_points: list[str] = Field(description="핵심 2-3개")

parser = PydanticOutputParser(pydantic_object=Summary)

prompt = ChatPromptTemplate.from_messages([
    ("system", "텍스트를 분석합니다. 한국어로.\n\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

chain_summary = prompt | llm | parser

r = chain_summary.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r)

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시일이 6월에서 7월로 연기됨']


## Phase 2. RunnableParallel — 여러 체인 결합

In [5]:
# 자연어 답장 체인 추가 (StrOutputParser)
chain_notice = (
    ChatPromptTemplate.from_messages([
        ("system", "팀원에게 알릴 한 줄 안내문을 작성하세요."),
        ("human", "{text}"),
    ]) | llm | StrOutputParser()
)

combined = RunnableParallel(summary=chain_summary, notice=chain_notice)

r = combined.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
# chain 결과 = 딕셔너리 {'summary' : (값), 'notice' : (값)}
print(r["summary"])
print(r["notice"])

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시일이 6월에서 7월로 연기됨']
신제품 출시가 6월에서 7월로 연기되었음을 알려드립니다.


## Phase 3. RunnableLambda — 함수도 체인에 (결과는 Pydantic으로)

In [6]:
class Decision(BaseModel):
    urgent: bool
    note: str

def route(x) -> Decision:
    s = x["summary"]
    return Decision(urgent=(s.sentiment == "negative"), note=s.title)

chain_routed = combined | RunnableLambda(route)

d = chain_routed.invoke({"text": "신제품 출시 연기"})
print(d)

urgent=True note='신제품 출시 연기'


## Phase 4. RunnablePassthrough — 원본 보존

In [7]:
full = RunnableParallel(
    summary=chain_summary,
    notice=chain_notice,
    original=RunnablePassthrough(),
)
r = full.invoke({"text": "신제품 출시 연기"})
print(r["original"])  # 입력 그대로 통과

{'text': '신제품 출시 연기'}


---
# 📰 Project. 뉴스 모니터링 + PR 대응 시스템

```
기사 → 분석 + 임원 요약 + 태깅 (동시)
     → 대응 라우팅 (즉시 알림 / 일일 브리핑 / 보관)
```

**모니터링 대상**: NovaTech (가상 AI 스타트업, B2B 챗봇 솔루션)

In [8]:
SAMPLE = """[테크크런치] NovaTech, B2B 챗봇 시장 점유율 1위 등극.
LLM 챗봇을 금융·제조 대기업 50여 곳에 공급하며 전년 대비 200% 성장.
다만 일부 고객사는 응답 속도 개선이 필요하다고 지적."""

# 분석 체인 — 자사 관점에서
class NewsAnalysis(BaseModel):
    headline: str = Field(description="15자 이내 헤드라인")
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    impact_score: int = Field(ge=0, le=10, description="자사 영향도 0-10")
    contains_risk: bool = Field(description="평판/사업 리스크 언급 여부")

p1 = PydanticOutputParser(pydantic_object=NewsAnalysis)
chain_analysis = (
    ChatPromptTemplate.from_messages([
        ("system", "자사(NovaTech, B2B AI 챗봇 스타트업) 관점에서 기사를 분석합니다.\n\n{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p1.get_format_instructions())
    | llm | p1
)
print(chain_analysis.invoke({"article": SAMPLE}))

headline='NovaTech, B2B 챗봇 시장 1위' sentiment='positive' impact_score=8 contains_risk=True


In [9]:
# 임원 보고용 3문장 요약 (자연어)
chain_brief = (
    ChatPromptTemplate.from_messages([
        ("system", "임원 보고용 *정확히 3문장* 요약: 사실 / 자사 관련성 / 시사점."),
        ("human", "{article}"),
    ]) | llm | StrOutputParser()
)
print(chain_brief.invoke({"article": SAMPLE}))

NovaTech는 B2B 챗봇 시장에서 1위로 등극하며, LLM 챗봇을 50여 개 금융 및 제조 대기업에 공급하여 전년 대비 200% 성장했다. 자사와 관련하여, 이러한 성장은 고객의 요구에 부응하는 혁신적인 솔루션 제공을 통해 이루어졌음을 보여준다. 그러나 응답 속도 개선이 필요하다는 고객의 피드백은 향후 서비스 개선의 중요한 시사점으로 작용할 수 있다.


In [11]:
# 태깅 체인 — 다중 선택 (list[Literal])
class NewsTags(BaseModel):
    topics: list[Literal["제품", "재무", "인사", "기술", "규제", "경쟁사", "기타"]]
    teams: list[Literal["PR", "마케팅", "프로덕트", "법무", "재무"]] = Field(
        description="이 기사를 봐야 할 사내 팀"
    )

p2 = PydanticOutputParser(pydantic_object=NewsTags)
chain_tags = (
    ChatPromptTemplate.from_messages([
        ("system", "기사에 적절한 주제와 관련 팀 태그.\n\n{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p2.get_format_instructions())
    | llm | p2
)

# 3개 체인 동시 실행
monitor_v1 = RunnableParallel(
    analysis=chain_analysis,
    brief=chain_brief,
    tags=chain_tags,
)

r = monitor_v1.invoke({"article": SAMPLE})
print(f"📌 {r['analysis'].headline} | 영향도 {r['analysis'].impact_score}")
print(f"🏷  {r['tags'].topics} / 팀: {r['tags'].teams}")

📌 NovaTech, B2B 챗봇 시장 1위 | 영향도 8
🏷  ['기술', '제품'] / 팀: ['프로덕트', '마케팅']


In [12]:
# 대응 라우팅 — RunnableLambda + Pydantic
class PRAction(BaseModel):
    priority: Literal["alert_now", "daily_brief", "archive"]
    channel: Literal["slack_urgent", "slack_daily", "notion"]
    message: str

def route_pr(x) -> PRAction:
    a = x["analysis"]
    if a.sentiment == "negative" and a.impact_score >= 7:
        return PRAction(priority="alert_now", channel="slack_urgent",
                       message=f"🚨 즉시 대응: {a.headline}")
    if a.impact_score >= 4:
        return PRAction(priority="daily_brief", channel="slack_daily",
                       message=f"📋 브리핑: {a.headline}")
    return PRAction(priority="archive", channel="notion",
                   message=f"📁 보관: {a.headline}")

monitor_v2 = monitor_v1 | RunnableLambda(route_pr)
print(monitor_v2.invoke({"article": SAMPLE}))

priority='daily_brief' channel='slack_daily' message='📋 브리핑: NovaTech, B2B 챗봇 시장 1위'


In [13]:
# 여러 기사 batch — 매일 들어오는 수십 건
ARTICLES = [
    {"article": SAMPLE},
    {"article": "NovaTech 일부 고객사 데이터 유출 의혹. 회사는 부인. 보안 업계 우려 제기."},
    {"article": "정부, AI 산업 5조원 지원금 편성. 스타트업·연구기관 대상."},
]
for a in monitor_v2.batch(ARTICLES):
    icon = {"alert_now": "🚨", "daily_brief": "📋", "archive": "📁"}[a.priority]
    print(f"{icon} {a.message}")

📋 📋 브리핑: NovaTech, B2B 챗봇 시장 1위
🚨 🚨 즉시 대응: 고객사 데이터 유출 의혹
📋 📋 브리핑: 정부, AI 산업 지원금 편성


---
## 🎓 마무리

같은 3개 부품으로 PR 모니터링 시스템 완성. **도메인이 달라도 패턴은 동일**.

**도전과제**: SNS 카피 체인 / 영문 번역 체인 / 경쟁사 비교 체인 추가

### 3번째 : 경쟁사 비교 체인 추가

In [ ]:
# 비교 대상 기사 — 자사 / 경쟁사 한 쌍
OUR_ARTICLE = SAMPLE  # 위에서 쓴 NovaTech 기사 재사용
RIVAL_ARTICLE = """[전자신문] DialogAI, 응답속도 0.8초 신형 챗봇 엔진 공개.
기존 대비 가격 30% 인하를 내세워 중견기업 시장 공략에 나섰다.
다만 금융권 레퍼런스는 아직 10여 곳에 그쳐 대기업 시장 진출은 시간이 필요하다는 평가."""

pair = {"our_article": OUR_ARTICLE, "rival_article": RIVAL_ARTICLE}  # 체인 입력은 기사 2개

In [14]:
import os

from dotenv import load_dotenv
load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

✅ API 키 OK


In [ ]:
# import 전역변수
# 라이브러리, 인터페이스, 모듈
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

In [ ]:
# 모델 정의
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

#### 비교 체인 설계

단순 복제(자사 체인 → 경쟁사 체인)가 아니라 **"같은 틀로 추출 → 나란히 비교 → 판정"** 구조로 만든다.

```
{our_article, rival_article}
   ├─ 자사 프로필 추출 ─┐  (같은 스키마 · 같은 체인 재사용)
   └─ 경쟁사 프로필 추출 ┘
        ↓ RunnableParallel
   프로필 보존(Passthrough) + 항목별 비교 체인 (동시)
        ↓
   RunnableLambda 로 대응 라우팅 (코드로 판정)
```

In [ ]:
# 공통 프로필 스키마 — 자사/경쟁사 양쪽에 똑같이 적용
class CompanyProfile(BaseModel):
    company: str
    positioning: str = Field(description="한 줄 포지셔닝")
    strengths: list[str] = Field(description="기사에 근거한 강점 1-3개")      # 문자열 리스트
    weaknesses: list[str] = Field(description="기사에 근거한 약점 1-3개")
    key_numbers: list[str] = Field(description="기사에 언급된 수치 (점유율, 속도, 가격 등)")
    recent_move: str = Field(description="최근 전략적 움직임")

p_profile = PydanticOutputParser(pydantic_object=CompanyProfile)

# 프로필 추출 체인 — {company} 만 바꿔서 양쪽에 재사용
chain_profile = (
    ChatPromptTemplate.from_messages([
        ("system", "기사에서 {company}의 프로필만 추출합니다. 기사에 없는 내용은 추측하지 마세요.\n\n{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p_profile.get_format_instructions())   # 스키마 설명을 프롬프트에 주입
    | llm | p_profile
)

In [ ]:
# 양쪽 프로필 동시 추출 — 같은 체인에 입력만 다르게 연결
def side(company: str, key: str):
    # 입력 dict 에서 해당 기사만 꺼내 chain_profile 이 받는 형식으로 변환
    return RunnableLambda(lambda x: {"company": company, "article": x[key]}) | 

profiles = RunnableParallel(                       # 두 체인을 동시에 실행하는 부품
    ours = side("NovaTech", "our_article"),
    rival = side("DialogAI", "rival_article"),            # 경쟁사 이름, 경쟁사 기사 키
)

r = profiles.invoke(pair)
print("자사 강점:", r["ours"].strengths)
print("경쟁사 강점:", r["rival"].strengths)

In [ ]:
# 비교 스키마 — 고정된 축별로 우열 판정
class AxisCompare(BaseModel):
    axis: Literal["시장지위", "기술/성능", "가격", "고객기반"]
    winner: Literal["자사", "경쟁사", "등급", "판단불가"] = Field(Description = "판단불가 - 근거 부족한 경우") # 자사 / 경쟁사 / 동급 / 판단불가 (근거 부족 대비)
    reason: str = Field(description="양쪽 프로필에 근거한 한 줄 사유")

class Comparison(BaseModel):
    axes: list[AxisCompare['axis']] = Field(description="4개 축 모두 포함")        # 위에서 만든 스키마의 리스트
    threat_level: Literal["low", "medium", "high"] = Field(description="경쟁사가 자사에 주는 위협도")  # low / medium / high
    our_edge: str = Field(description="자사의 우위 한 줄")
    our_gap: str = Field(description="자사의 보완점 한 줄")
    counter_strategy: str = Field(description="대응 전략 한 줄")

p_cmp = PydanticOutputParser(pydantic_object=Comparison)

# 비교 체인 — 두 프로필을 JSON 으로 나란히 넣어 비교
chain_compare = (
    RunnableLambda(lambda x: {
        "ours": x["ours"].JsonOutputParser(indent=2),     # Pydantic 객체 → JSON 문자열
        "rival": x["rival"].JsonOutputParser(indent=2),
    })
    | ChatPromptTemplate.from_messages([
        ("system", "자사(NovaTech) 관점에서 경쟁사와 비교합니다. 두 프로필에 있는 내용만 근거로 하세요.\n\n{format_instructions}"),
        ("human", "[자사]\n{ours}\n\n[경쟁사]\n{rival}"),
    ]).partial(format_instructions=p_cmp.get_format_instructions())
    | llm | p_cmp
)

In [ ]:
# 비교 파이프라인 v1 — 프로필 추출 → (프로필 보존 + 비교) 동시
compare_v1 = chain_profile | RunnableParallel(
    profiles=RunnablePassthrough(),          # 추출된 두 프로필을 그대로 통과 (LLM 재호출 없음)
    comparison=chain_compare,        # 두 프로필을 받아 비교하는 체인
)

r = compare_v1.invoke(pair)
for a in r["comparison"].axes:
    print(f"[{a.axis}] {a.winner} — {a.reason}")
print("위협도:", r["comparison"].threat_level)

In [ ]:
# 대응 라우팅 — 비교 결과를 코드로 판정
class CompAction(BaseModel):
    priority: Literal["strategy_alert", "watch", "archive"]
    channel: Literal["slack_urgent", "slack_daily", "notion"]
    message: str

def route_compare(x) -> CompAction:
    c = x[____]                                                   # 비교 결과가 담긴 키
    lost = sum(1 for a in c.axes if a.winner == ____)             # 경쟁사가 앞선 축 개수
    if c.threat_level == ____ or lost >= 2:
        return CompAction(priority="strategy_alert", channel="slack_urgent",
                          message=f"🚨 경쟁사 대응 필요 (열세 {lost}개 축): {c.counter_strategy}")
    if c.threat_level == ____ or lost == 1:
        return CompAction(priority="watch", channel="slack_daily",
                          message=f"👀 경쟁사 동향 (열세 {lost}개 축): {c.our_gap}")
    return CompAction(priority="archive", channel="notion",
                      message=f"📁 보관: {c.our_edge}")

compare_v2 = compare_v1 | ____(route_compare)    # 함수를 체인 부품으로
print(compare_v2.invoke(pair))

In [ ]:
# 여러 쌍 batch — 경쟁사 기사만 바꿔서 비교
PAIRS = [
    pair,
    {"our_article": OUR_ARTICLE,
     "rival_article": "DialogAI, 대규모 서비스 장애로 고객사 3곳 이탈. 시장 점유율은 5% 수준에 머물러."},
]
for p, a in zip(PAIRS, compare_v2.____(PAIRS)):    # 여러 입력을 한 번에
    print(a.message)